# 01 - Data Preparation: Multi-Component RUL + Fault Classification (FD004)

In [39]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

In [40]:
# Simulate sensor + RUL data
np.random.seed(42)
n_units = 5
n_cycles = 50
sensor_cols = [f'sensor{i}' for i in range(1, 22)]

records = []
for unit in range(1, n_units + 1):
    for cycle in range(1, n_cycles + 1):
        sensors = np.random.normal(loc=100, scale=10, size=len(sensor_cols))
        record = [unit, cycle] + list(sensors)
        records.append(record)

df = pd.DataFrame(records, columns=['unit', 'cycle'] + sensor_cols)
df.head()

,unit,cycle,sensor1,sensor2,sensor3,sensor4,sensor5,sensor6,sensor7,sensor8,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,104.967142,98.617357,106.476885,115.230299,97.658466,97.658630,115.792128,107.674347,...,95.342702,102.419623,80.867198,82.750822,94.377125,89.871689,103.142473,90.919759,85.876963,114.656488
1,1,2,97.742237,100.675282,85.752518,94.556173,101.109226,88.490064,103.756980,93.993613,...,99.865028,89.422891,108.225449,87.791564,102.088636,80.403299,86.718140,101.968612,107.384666,101.713683
2,1,3,98.843517,96.988963,85.214780,92.801558,95.393612,110.571222,103.436183,82.369598,...,106.116763,110.309995,109.312801,91.607825,96.907876,103.312634,109.755451,95.208258,98.143410,88.936650
3,1,4,88.037934,108.125258,113.562400,99.279899,110.035329,103.616360,93.548802,103.613956,...,73.802549,108.219025,100.870471,97.009926,100.917608,80.124311,97.803281,103.571126,114.778940,94.817298
4,1,5,91.915064,94.982430,109.154021,103.287511,94.702398,105.132674,100.970775,109.686450,...,85.364851,102.961203,102.610553,100.051135,97.654129,85.846293,95.793547,96.572855,91.977227,98.387143


In [41]:
# Simulate RUL for 3 components
component_map = {
    'comp1': sensor_cols[1:7],
    'comp2': sensor_cols[7:14],
    'comp3': sensor_cols[14:21]
}
components = list(component_map.keys())

rul_df = df.groupby('unit').agg({'cycle': 'max'}).rename(columns={'cycle': 'max_cycle'}).reset_index()
df = df.merge(rul_df, on='unit')
for comp in components:
    df[f'{comp}_rul'] = df['max_cycle'] - df['cycle'] + np.random.randint(-5, 5, len(df))
df.head()

,unit,cycle,sensor1,sensor2,sensor3,sensor4,sensor5,sensor6,sensor7,sensor8,...,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21,max_cycle,comp1_rul,comp2_rul,comp3_rul
0,1,1,104.967142,98.617357,106.476885,115.230299,97.658466,97.658630,115.792128,107.674347,...,94.377125,89.871689,103.142473,90.919759,85.876963,114.656488,50,50,52,44
1,1,2,97.742237,100.675282,85.752518,94.556173,101.109226,88.490064,103.756980,93.993613,...,102.088636,80.403299,86.718140,101.968612,107.384666,101.713683,50,44,44,44
2,1,3,98.843517,96.988963,85.214780,92.801558,95.393612,110.571222,103.436183,82.369598,...,96.907876,103.312634,109.755451,95.208258,98.143410,88.936650,50,43,51,42
3,1,4,88.037934,108.125258,113.562400,99.279899,110.035329,103.616360,93.548802,103.613956,...,100.917608,80.124311,97.803281,103.571126,114.778940,94.817298,50,44,42,50
4,1,5,91.915064,94.982430,109.154021,103.287511,94.702398,105.132674,100.970775,109.686450,...,97.654129,85.846293,95.793547,96.572855,91.977227,98.387143,50,48,49,49


In [42]:
# Simulate fault labels: fault = 1 if RUL < 20
for comp in components:
    df[f'{comp}_fault'] = (df[f'{comp}_rul'] < 20).astype(int)
df[[f'{comp}_rul' for comp in components] + [f'{comp}_fault' for comp in components]].head()

,comp1_rul,comp2_rul,comp3_rul,comp1_fault,comp2_fault,comp3_fault
0,50,52,44,0,0,0
1,44,44,44,0,0,0
2,43,51,42,0,0,0
3,44,42,50,0,0,0
4,48,49,49,0,0,0


In [43]:
# Build supervised learning data: 30-sequence sensor input → output RUL and fault labels
sequence_length = 30
X, y_reg, y_cls = [], [], []

for unit in df['unit'].unique():
    df_unit = df[df['unit'] == unit].sort_values('cycle')
    data = df_unit[sensor_cols].values
    for i in range(sequence_length, len(df_unit)):
        X.append(data[i-sequence_length:i])
        y_reg.append(df_unit.iloc[i][[f'{comp}_rul' for comp in components]].values)
        y_cls.append(df_unit.iloc[i][[f'{comp}_fault' for comp in components]].values)

X = np.array(X)
y_reg = np.array(y_reg)
y_cls = np.array(y_cls)
X.shape, y_reg.shape, y_cls.shape

((100, 30, 21), (100, 3), (100, 3))

In [44]:
# Save data to outputs
os.makedirs('outputs', exist_ok=True)
np.save('../outputs/X.npy', X)
np.save('../outputs/y_reg.npy', y_reg)
np.save('../outputs/y_cls.npy', y_cls)

print(f'Shape of input sequences: {X.shape}')
print(f'Shape of RUL targets: {y.shape}')

Shape of input sequences: (100, 30, 21)
Shape of RUL targets: (53779, 3)
